### 專案報告：台灣產業勞動結構量化診斷 (2013-2024)
#### 一、 專案背景與目的 (Project Background & Purpose)
在就業市場中，求職者往往僅能透過「職稱」與「薪資」想像工作環境，卻難以看透產業的「勞動骨架」。本專案旨在解決求職者與產業間的資訊不對稱，透過 Python 交叉分析過去10年間（2013-2024）台灣 38 個主要產業別的官方數據，分析各產業的勞動結構。

#### 「就業」 和「受僱」的定義
1. 就業 (Employment)
**定義： 指個人投入社會生產活動，以獲取報酬或經營家屬事業之狀態。**
廣義範疇：在統計與法律實務上，只要從事有酬工作（或每週工作 15 小時以上之無酬家屬工作）皆視為就業。
包含對象：
* 雇主：自行經營事業並僱用他人者。
* 自營作業者：獨自經營事業（如個人工作室、自由接案者）。
* 受僱者：受領薪資並受他人指揮者。
* 無酬家屬工作者：幫家屬經營事業且達到法定時數者。

2. 受僱 (Employed)
**定義： 指受雇主僱用，在他人指揮監督下從事工作，並藉此獲取工資或報酬之行為。**


本專案使用的資料為「經濟部中小及新創企業署」於政府公開資料平台提供的「年度中小企業之行業與各項指標值及比率」資料。(2013年至2024年的資料)
透過 OS、SR、LR 三大核心指標，揭示產業在體系穩定度、僱用純度與職涯天花板的狀況，協助求職者進行「結構性避險」。

#### 二、 核心分析指標說明 (Core Indicators)
本模型以「產業別」為變數，套入以下核心指標公式。
【本報告後續提及之所有受僱人數、企業家數、就業人數等數值，均採 2013-2024 10年期間之算術平均值進行運算。】
##### 1. 平均組織規模 ($OS$) —— 判斷「體系穩定度」公式：$$OS = \frac{\text{受僱人數 (人)}}{\text{企業家數 (家)}}$$
【判斷標準】
* 高 OS ($> 15$): 屬於「大型組織型」。代表該產業以大廠、大型體系為主，制度化程度高，勞動環境受法規保障較健全，結構穩定。
* 低 OS ($< 5$): 屬於「微型碎片型」。勞動力分散在極小單位，工作環境高度受雇主個人特質影響，結構穩定度較低。
##### 2. 自營與非典型就業率 ($SR$) —— 判斷「僱用純度」公式：$$SR = \frac{\text{總就業人數} - \text{受僱人數}}{\text{總就業人數}}$$
【判斷標準】
* 高 SR ($> 20\%$): 屬於「高自營/家族化」。代表產業中非典型勞動（自營者、家屬）佔比高，求職者需注意傳統受僱福利保障可能較不完善。
* 低 SR ($< 10\%$): 屬於「高純度僱用」。產業幾乎由領薪水的標準受僱勞工組成，勞務關係單純明確。
##### 3. 大企業依賴度 ($LR$) —— 判斷「職涯天花板」公式：$$LR = \frac{\text{大企業受僱人數}}{\text{受僱總人數}}$$
【判斷標準】
* 高 LR ($> 30\%$): 屬於「資源集中型」。大企業吸納力強，通常具備完善的內部晉升階梯與教育訓練。
* 低 LR ($< 10\%$): 屬於「扁平競爭型」。缺乏大型組織支撐，管理職缺相對稀缺，求職者可能面臨較明顯的職涯發展瓶頸。

#### 三、 分析做法 (Analysis Workflow)
本專案的完整分析流程如下：








5. **資料篩選與排序**：使用 `sort_values()` 與 `query()` / 布林索引，依指標大小篩選出各類型代表性產業。

6. **繪製動態折線圖**：
   - 使用 **`plotly`**（繪製互動式折線圖，呈現各產業指標於 2020–2024 年的逐年趨勢。



| 步驟 | 說明 |
| :--- | :--- |
| **1. 載入套件** | `pandas` + `plotly` |
| **2. 讀取資料** | `pd.read_csv()` 讀入就業人數、受雇人數兩份 CSV |
| **3. 資料提取與定義運算變數** | 把「運算指標會用到的欄位」抽取出來，並定義成單獨的變數。利用 `groupby('行業別').mean()` 算出各產業涵蓋 2013-2024 年（十年間）的平均數|
| **4. 指標運算** | 計算 OS、SR、LR，以 `groupby()` 彙整五年平均 |
| **5. 資料篩選與排序** | `sort_values()`、`query()` 篩選代表性產業 |
| **6. 繪製動態折線圖** | 使用 `plotly` 畫互動式趨勢折線圖 |


#### 步驟 1：載入套件
匯入 `pandas` 作為資料處理核心，搭配 `plotly.express` / `plotly.graph_objects` 作為動態視覺化工具。

In [2]:
import pandas as pd               # 資料處理核心套件
import plotly.express as px       # Plotly 動態圖表（快速繪圖）
import plotly.graph_objects as go # Plotly 動態圖表（進階客製化）

#### 步驟2：讀取資料
使用 `pd.read_csv()` 分別讀取 **「企業行業別就業人數統計」** 與 **「企業行業別受僱人數統計」** 兩份 CSV 檔案，建立 `df_employment` 與 `df_hire` 兩個 data frame。

In [3]:
df_employment = pd.read_csv('企業行業別就業人數統計.csv') 
df_employed = pd.read_csv('企業行業別受僱人數統計.csv')
df_company = pd.read_csv('企業家數.csv')

print(f"✔️ 就業人數 資料筆數: {len(df_employment)}")
print(f"✔️ 受雇人數 資料筆數: {len(df_employed)}")
print(f"✔️ 企業家數 資料筆數:{len(df_company)}")


✔️ 就業人數 資料筆數: 228
✔️ 受雇人數 資料筆數: 228
✔️ 企業家數 資料筆數:222


#### 步驟3：資料提取與定義運算變數
- **提取核心欄位**：從原始資料庫中提取計算三大指標的四個數據作為變數：`總就業人數`、`總受僱人數`、`大企業受僱人數` 與 `企業家數`。
- **建置數值清洗工具**：數字有時會帶有千分位與特殊字元（如：逗號 `,`、連字號 `-`），透過自訂函式 `clean_num`，使用 `pd.to_numeric()` 搭配 `errors='coerce'`，將髒字串清洗為能進行數學運算的「純數字」，並把缺失值轉化為不影響運算的 `NaN`。
- **計算並建立全期變數 (`groupby`)**：利用 `groupby('行業別').mean()` 一口氣算出各產業涵蓋 2013-2024 年（十年間）的平均。
- **最終產出四大運算變數**：
  1. `avg_total_employment` (10年平均 - 總就業人數)
  2. `avg_total_employed` (10年平均 - 總受僱人數)
  3. `avg_large_corp_employed` (10年平均 - 大企業受僱人數)
  4. `avg_company_count` (10年平均 - 企業家數)

In [4]:
# 自定義函式：將帶有逗號或缺漏字串的欄位，洗乾淨轉為純數字
def clean_num(series):
    return pd.to_numeric(series.astype(str).str.replace(',', ''), errors='coerce')


# 1. 總就業人數 (十年期平均)
avg_total_employment = clean_num(df_employment['全部企業(千人)']).groupby(df_employment['行業別']).mean()

# 2. 總受僱人數 (十年期平均) 
avg_total_employed = clean_num(df_employed['全部企業(千人)']).groupby(df_employed['行業別']).mean()

# 3. 大企業受僱人數 (十年期平均)
avg_large_corp_employed = clean_num(df_employed['大企業(千人)']).groupby(df_employed['行業別']).mean()

# 4. 企業家數 (十年期平均)
avg_company_count = clean_num(df_company['全部企業(家)']).groupby(df_company['行業別']).mean()

print("【💼 十年平均總就業人數】:")
display(avg_total_employment[:5])


【💼 十年平均總就業人數】:


行業別
不動產業                 102.750000
住宿及餐飲業               836.500000
公共行政及國防；強制性社會安全      374.666667
其他服務業                555.166667
出版、影音製作、傳播及資通訊服務業    260.571429
Name: 全部企業(千人), dtype: float64

#### 步驟四：指標運算
完成變數提取後，在這一步將四個變數結合代數公式，運算出代表產業勞動體質的核心指標。
因為 Pandas 系列 (Series) 的索引對齊特性，計算時不需迴圈，只要將變數相加減，對應的「行業別」就會精準媒合。

**具體執行的公式與計算：**
1. **補算非受雇人數**： `avg_non_employed` = 總就業平均 $-$ 受僱總數平均
2. **計算 OS (體系穩定度)**： 受雇人數 $\div$ 企業家數。數值越大代表該產業愈集中於大體系之中
3. **計算 SR (僱用純度)**： 非受雇人數 $\div$ 總就業人數 $\times 100 (\%)$。藉此判斷家族化或接案自營的濃度。
4. **計算 LR (職涯天花板)**： 大企業受僱 $\div$ 總受僱 $\times 100 (\%)$。評估該產業中大企業吸納就業資源的佔比。
5. **整合分析總表 (`df_indicators`)**： 將所有運算完的指標自定義為新的 DataFrame，作為後續「資料篩選」與「動態繪圖」 的參照！


In [5]:
# 1. 計算「非受雇人數」(自營/老闆/無酬家屬) 
#    直接將 總就業平均 減去 受僱平均：
avg_non_employed = avg_total_employment - avg_total_employed

# 2. 計算 OS (平均組織規模) —— 判斷「體系穩定度」
#    公式：受僱總數(人) / 企業家數(家)
#    💡 注意：因為原本的受僱資料單位是「千人」，在除以家數之前要先乘上 1000 還原成真實人數
avg_os = (avg_total_employed * 1000) / avg_company_count

# 3. 計算 SR (自營與非典型就業率) —— 判斷「僱用純度」
#    公式：非受雇人數 / 總就業人數 * 100 (%)
avg_sr = (avg_non_employed / avg_total_employment) * 100

# 4. 計算 LR (大企業依賴度) —— 判斷「職涯天花板」
#    公式：大企業受僱人數 / 總受僱人數 * 100 (%)
avg_lr = (avg_large_corp_employed / avg_total_employed) * 100


# 5. 將運算完的指標自定義為新的 DataFrame（df_indicators）
df_indicators = pd.DataFrame({
    'OS': avg_os,   # 平均組織規模
    'SR': avg_sr,   # 自營僱用率 (%)
    'LR': avg_lr    # 大企業依賴度 (%)
}).reset_index()  # 用 reset_index 把行業別標籤拉回成正常的欄位

# 圓整到小數點後兩位，讓報表更乾淨
df_indicators = df_indicators.round(2)
# 讓表格最左側的編號從 1 開始算起
df_indicators.index = df_indicators.index + 1

print("【台灣產業勞動結構總表】（預覽前10個產業）")
display(df_indicators.head(10))  # 預覽前 10 個產業


【台灣產業勞動結構總表】（預覽前10個產業）


,行業別,OS,SR,LR
1,不動產業,2.25,9.16,2.32
2,住宿及餐飲業,3.28,34.78,2.89
3,公共行政及國防；強制性社會安全,NaN,0.00,0.00
4,其他服務業,3.84,39.04,0.98
5,出版、影音製作、傳播及資通訊服務業,10.11,5.65,21.27
6,出版影音及資通訊業,8.92,7.26,16.03
7,專業、科學及技術服務業,5.63,19.20,13.07
8,批發及零售業,1.64,36.32,4.15
9,支援服務業,8.24,7.43,6.70
10,教育服務業,326.68,5.03,10.43


#### 步驟五：指標篩選與排序 
- **極值排序 (`sort_values`)**：透過 `ascending=False` (由大到小遞減排序)，我們可以精準抓出「自營濃度最高(Top SR)」與「大企業主導比例最重(Top LR)」的極端產業，並用 `.head(5)` 取出前五名。
- **多條件過濾 (`query`)**：當我們想要尋找「制度化且規模大」的特例產業時，可以直接使用 `query()` 輸入字串條件（如要求 OS 大於 15 人 且 LR 大於 30%）。
- **編號校正 (`.index = range(...)`)**：在排序與篩選後，原本的 Index 會跟著被打亂，因此我們重新賦予它們從 1 開始的新編號，以還原整潔的報表外觀。

In [6]:
# 1. df_top_sr: 存放依據 SR (自營僱用率) 降冪排序的前 5 名
#    找尋「高自營 / 家族化」的代表產業
df_top_sr = df_indicators.sort_values(by='SR', ascending=False).head(5)
df_top_sr.index = range(1, len(df_top_sr) + 1)

# 2. df_top_lr: 存放依據 LR (大企業依賴度) 降冪排序的前 5 名
#    找尋「大企業壟斷、職涯天花板高」的代表產業
df_top_lr = df_indicators.sort_values(by='LR', ascending=False).head(5)
df_top_lr.index = range(1, len(df_top_lr) + 1)

# 3. df_big_system: 存放符合雙重篩選條件，符合「穩健大型體系」的產業
#   「穩健大型體系」的代表產業指標： OS 超過 15 人，且 LR 超過 30%
query_condition = 'OS > 15 & LR > 30'
df_big_system = df_indicators.query(query_condition).copy()
df_big_system.index = range(1, len(df_big_system) + 1)

# 成果展示 (Display Results)
print("🏅 前 5 大「高自營/家族化」產業（Top 5 Highest SR）:") # SR(%) 最高的前 5 大「高自營/家族化」產業
display(df_top_sr)

print("\n🏢 前 5 大「大企業壟斷」產業（Top 5 Highest LR:）") # LR(%) 最高的前 5 大「大企業壟斷」產業
display(df_top_lr)

print("\n⚙️ 穩定的「大型體系」產業（Large & Stable Systems (OS > 15 & LR > 30):）") # 穩定的「大型體系」產業
display(df_big_system)


🏅 前 5 大「高自營/家族化」產業（Top 5 Highest SR）:


,行業別,OS,SR,LR
1,農、林、漁、牧業,7.98,82.98,0.89
2,其他服務業,3.84,39.04,0.98
3,批發及零售業,1.64,36.32,4.15
4,住宿及餐飲業,3.28,34.78,2.89
5,運輸及倉儲業,10.35,21.39,16.32



🏢 前 5 大「大企業壟斷」產業（Top 5 Highest LR:）


,行業別,OS,SR,LR
1,資訊及通訊傳播業,11.91,5.69,31.32
2,製造業,18.83,7.34,30.48
3,醫療保健及社會工作服務業,336.78,5.41,28.42
4,出版、影音製作、傳播及資通訊服務業,10.11,5.65,21.27
5,運輸及倉儲業,10.35,21.39,16.32



⚙️ 穩定的「大型體系」產業（Large & Stable Systems (OS > 15 & LR > 30):）


,行業別,OS,SR,LR
1,製造業,18.83,7.34,30.48


#### 步驟六：繪製產業動態趨勢折線圖 
選用 `Plotly` 模組，能夠在畫面中渲染出流暢互動的動態折線圖。
- **歷年資料庫重建**：因為前述步驟計算的是「10 年全期大平均」，這裡重新使用 `pd.merge()` 對齊三份原始報表，算出各行業 2013-2024 年「每一年的 SR、LR、OS 數值」，作為繪圖的 X 軸與 Y 軸基礎。
- **客製化繪圖函式 (`plot_trend`)**：自定義專用的畫圖函式。只要傳入「產業名單」與「想觀察的指標」，就會自動生成折線圖。
- **完美版面設定 (`update_layout`)**：為了讓圖表的視野最大化，將圖例 (Legend) 改為水平排列 (`orientation="h"`) 並移至圖表正下方，把左右空間完全還給折線圖。
- **生成三張關鍵報表**：
   - 🟢 圖一：觀察高自營/家族化產業的「SR (自營率)」變動。
   - 🔵 圖二：觀察大企業壟斷產業的「LR (大企業佔比)」變動。
   - 🟣 圖三：觀察穩定大型體系的「OS (平均每家員工人數)」變革。

> 💡 **互動提示：** 指標生成後，您可以將游標懸浮於折線上的小圓點查看確切數字，或是點擊圖表下方的「產業名稱」來單獨隱藏/顯示特定產業的線條。


In [9]:
import plotly.express as px

# 1. 重建歷年資料庫：為了能畫出任意產業的歷年折線，我們先建立一個包含「所有年份、所有產業」的總表
#    將原始三張表的數字洗乾淨
df_employment['val'] = pd.to_numeric(df_employment['全部企業(千人)'].astype(str).str.replace(',', ''), errors='coerce')
df_employed['val_total'] = pd.to_numeric(df_employed['全部企業(千人)'].astype(str).str.replace(',', ''), errors='coerce')
df_employed['val_large'] = pd.to_numeric(df_employed['大企業(千人)'].astype(str).str.replace(',', ''), errors='coerce')
df_company['val_comp'] = pd.to_numeric(df_company['全部企業(家)'].astype(str).str.replace(',', ''), errors='coerce')

#    用 merge 精準合併三張表的歷年數據
df_trend = pd.merge(df_employment[['年度', '行業別', 'val']], 
                    df_employed[['年度', '行業別', 'val_total', 'val_large']], on=['年度', '行業別'])
df_trend = pd.merge(df_trend, 
                    df_company[['年度', '行業別', 'val_comp']], on=['年度', '行業別'])

#    計算歷年每個產業的 OS、SR、LR
df_trend['SR'] = ((df_trend['val'] - df_trend['val_total']) / df_trend['val']) * 100
df_trend['LR'] = (df_trend['val_large'] / df_trend['val_total']) * 100
df_trend['OS'] = (df_trend['val_total'] * 1000) / df_trend['val_comp']


# 2. 定義繪圖專用機器 (小函式)：只要丟知名單、告訴要畫什麼指標，它就會自動生成圖表！
def plot_trend(target_list, y_col, y_title, chart_title):
    # 篩選名單
    df_plot = df_trend[df_trend['行業別'].isin(target_list)]
    
    # 使用 Plotly 畫線
    fig = px.line(
        df_plot, x='年度', y=y_col, color='行業別', markers=True,
        labels={'年度': '年份', y_col: y_title}, title=chart_title
    )
    
    # ✅ 調整圖表版面 (重點新增區塊)
    fig.update_layout(
        xaxis=dict(type='category'),   # 強制年份一格一格完美顯示
        legend=dict(
            orientation="h",           # h 代表水平排列 (Horizontal)
            yanchor="top",             # 對齊基準點設為上方
            y=-0.2,                    # 把它塞到圖表(Y軸的0點)下方，留出一點空白 (-0.2)
            xanchor="center",          # X軸對齊基準點為置中
            x=0.5,                     # 放在畫面正中央
            title=None                 # 隱藏 "行業別" 這個圖例標題以節省更多空間
        ),
        margin=dict(b=100)             # 在最下面預留 100px 的空間，避免圖例被切掉
    )
    
    fig.show()

    
# 3. 施放魔法：一口氣畫出你要的 3 張圖表！

# 📈 圖表 1：前 5 大「高自營 / 家族化」產業 (觀察 SR 指標)
plot_trend(
    target_list=df_top_sr['行業別'].tolist(),
    y_col='SR',
    y_title='SR (%) 自營率',
    chart_title='🟢 圖一：前 5 大「高自營 / 家族化」產業歷年 SR 變動趨勢'
)

# 📈 圖表 2：前 5 大「大企業壟斷」產業 (觀察 LR 指標)
plot_trend(
    target_list=df_top_lr['行業別'].tolist(),
    y_col='LR',
    y_title='LR (%) 大企業佔比',
    chart_title='🔵 圖二：前 5 大「大企業壟斷」產業歷年 LR 變動趨勢'
)

# 📈 圖表 3：穩定的「大型體系」產業 (觀察 OS 平均組織規模)
plot_trend(
    target_list=df_big_system['行業別'].tolist(),
    y_col='OS',
    y_title='OS (平均每家員工人數)',
    chart_title='🟣 圖三：穩定「大型體系」(OS>15 且 LR>30) 產業歷年規模變革'
)
